# 1. Imports

In [ ]:
# ==================== IMPORTS ====================
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

# ==================== GLOBAL CONFIGURATION ====================
RANDOM_STATE = 42

# 2. Load Data

In [2]:
# Load data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Normalize column names
train.columns = train.columns.str.lower()
test.columns = test.columns.str.lower()

# Extract and remove Test ID
test_id = test.pop("id") if "id" in test.columns else test.index.copy()

# Remove Train ID
if "id" in train.columns:
    train = train.drop(columns=["id"])

# 3. Data Validation Report

In [114]:
# DATA preprocessing checklist:
# - COLUMN ALIGNMENT +
# - DUPLICATE ROWS + 
# - MISSING VALUES (NAN) +
# - INCORRECT VALUES + num - object
# - FEATURE SCALING
# - ENCODING
# - INCORRECT DATA TYPES
# - UNEXPECTED TARGET VALUES
# Do full diagnostic report (last section) in the end

# ============================================================
# DATA VALIDATION REPORT
# Pipeline checks: column alignment, missing values, 
# duplicates, numeric outliers + boxplots,
# ============================================================

# ---- Report configuration ----
WIDTH = 140            # line width for report output
target = "personality" # Target column (expected: Train-only)

# ---- Column alignment check ----
# Counter preserves duplicate columns that set() would silently drop,
# so mismatches in column *counts* are caught, not just column names.
train_only = Counter(train.columns) - Counter(test.columns)
test_only = Counter(test.columns) - Counter(train.columns)

# ============================================================
# [1] DATA OVERVIEW
# ============================================================
print("=" * WIDTH)
print("DATA INFORMATION REPORT".center(WIDTH))
print("=" * WIDTH)

print("\n" + "[1] DATA OVERVIEW")
print("-" * WIDTH)

# Shape tells us sample size and feature count at a glance
print(f"{'Training dataset shape':<30}: {train.shape}")
print(f"{'Test dataset shape':<30}: {test.shape}")

# Explicit feature counts - catches duplicate column names
# that would inflate train.shape[1] invisibly
print(f"{'Training features':<30}: {len(train.columns)}")
print(f"{'Test features':<30}: {len(test.columns)}")

# ============================================================
# [2] COLUMNS INFORMATION
# ============================================================
print("\n" + "[2] COLUMN INFORMATION")
print("-" * WIDTH)

print(f"{'Training columns':<30}: {train.columns.tolist()[:15]}")
print(f"{'Test columns':<30}: {test.columns.tolist()[:15]}")

# ============================================================
# [3] COLUMN VALIDATION
# Expectation: identical features in both sets,
# ============================================================
print("\n" + "[3] COLUMN VALIDATION")
print("-" * WIDTH)

print(f"{'Features exclusive to Train':<30}: {dict(train_only) or '{none}'}")
print(f"{'Features exclusive to Test':<30}: {dict(test_only) or '{none}'}")

# ============================================================
# [4] DUPLICATES
# duplicated() marks copies AFTER the first occurrence: the first is the
# "original", everything flagged here is an extra copy to investigate.
# ============================================================
print("\n" + "[4] DUPLICATES")
print("-" * WIDTH)

train_dup_mask = train.columns.duplicated()
test_dup_mask = test.columns.duplicated()

# Row duplicates overweight those samples during training - count only,
# materializing the rows here would waste memory for a number.
train_dup_cnt = train.duplicated().sum()
test_dup_cnt = test.duplicated().sum()

if not train_dup_mask.any() and not test_dup_mask.any():
    print("STATUS: PASS - no duplicate columns detected 🟢")
else:
    print(f"{'Train duplicate columns:':<30}: {train.columns[train_dup_mask].tolist()}")
    print(f"{'Test duplicate columns:':<30}: {test.columns[test_dup_mask].tolist()}")

if train_dup_cnt == 0 and test_dup_cnt == 0:
    print("\n" + "STATUS: PASS - no duplicate rows detected 🟢")
else:
    print(f"{'Train duplicate rows:':<30}: {train_dup_cnt}")
    print(f"{'Test duplicate rows:':<30}: {test_dup_cnt}")

# ============================================================
# [5] MISSING VALUES
# Per-column missing counts and percentage for both datasets,
# sorted so the most damaged column is always on top.
# ============================================================
print("\n" + "[5] MISSING VALUES")
print("-" * WIDTH)

# Count and percentage per column, for both datasets
train_missing_cnt = train.isnull().sum()
train_missing_pct = train.isnull().mean() * 100
test_missing_cnt = test.isnull().sum()
test_missing_pct = test.isnull().mean() * 100

missing_report = pd.DataFrame({
    'train_cnt': train_missing_cnt,
    'train_pct': train_missing_pct.round(2),
    'test_cnt':  test_missing_cnt,
    'test_pct':  test_missing_pct.round(2)
})

# Sort by the worst side of each column: sorting by train alone would
# bury a column that's clean in train but heavily missing in test.
missing_report['worst_pct'] = missing_report[['train_pct', 'test_pct']].max(axis=1)
missing_report = missing_report.sort_values('worst_pct', ascending=False)
missing_report = missing_report.drop(columns='worst_pct')

if train_missing_cnt.sum() == 0 and test_missing_cnt.sum() == 0:
    print("STATUS: PASS - no missing values detected 🟢")
else:
    print(missing_report.to_string())
    
# ============================================================
# [6] NUMERIC OUTLIERS (IQR fence)
# Fence computed on TRAIN only, applied to both datasets.
# ============================================================
print("\n" + "[6] NUMERIC OUTLIERS")
print("-" * WIDTH)

train_num_cols = train.select_dtypes(include=np.number).columns
test_num_cols = test.select_dtypes(include=np.number).columns

diff_cols = train_num_cols.symmetric_difference(test_num_cols)
if len(diff_cols) > 0:
    print("STATUS: WARNING - numeric column sets differ between train and test 🟡")
    print(f"  non-matching columns: {list(diff_cols)}\n")

# Only columns existing on BOTH sides can be compared - shrink to the
# common set, the mismatch was reported above (see section [3]).
num_cols = train_num_cols.intersection(test_num_cols, sort=False)

if len(num_cols) == 0:
    print("STATUS: PASS - no common numeric columns - section skipped 🟢")
else:
    # Compute the fence for all numeric columns in four vectorized lines.
    # q1/q3/iqr are Series - one value per column, no loops.
    q1 = train[num_cols].quantile(0.25)
    q3 = train[num_cols].quantile(0.75)
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    # Boolean mask: True where a value sits outside the fence.
    # One train-learned fence, held against both datasets - same ruler,
    # fair comparison.
    # NaN cells become False automatically - they belong to section [5].
    outside_fence_train = (train[num_cols] < lower_bound) | (train[num_cols] > upper_bound)
    outlier_cnt_train = outside_fence_train.sum()
    
    outside_fence_test = (test[num_cols] < lower_bound) | (test[num_cols] > upper_bound)
    outlier_cnt_test = outside_fence_test.sum()
    
    # Report: violation count, fence, and actual min/max for inspection.
    # min/max let you judge "impossible" vs "rare but real" without
    # extracting rows - the report answers, the human decides.
    outlier_report = pd.DataFrame({
        'train_cnt':  outlier_cnt_train,
        'test_cnt':   outlier_cnt_test,
        'train_min':  train[num_cols].min(),
        'train_max':  train[num_cols].max(),
        'test_min':   test[num_cols].min(),
        'test_max':   test[num_cols].max(),
        'fence_low':  lower_bound.round(2),
        'fence_high': upper_bound.round(2),
    })
    
    # A column dirty on EITHER side makes the report; worst offender on top
    outlier_report = outlier_report[(outlier_report['train_cnt'] > 0) |
                                    (outlier_report['test_cnt'] > 0)]
    outlier_report['worst_cnt'] = outlier_report[['train_cnt', 'test_cnt']].max(axis=1)
    outlier_report = outlier_report.sort_values('worst_cnt', ascending=False)
    outlier_report = outlier_report.drop(columns='worst_cnt')

    if outlier_report.empty:
        print("STATUS: PASS - no numeric outliers detected 🟢")
    else:
        print(outlier_report.to_string())

    print("\n" + "Boxplots (train vs test)".center(WIDTH))
    print("-" * WIDTH)

    # Duplicated names break train[col] (returns a DataFrame, not a Series).
    # Report which names got collapsed; the full diagnosis lives in [3].
    dup_in_plot = num_cols[num_cols.duplicated()].tolist()
    if dup_in_plot:
        print(f"STATUS: WARNING - duplicated names plotted once (first copy): {dup_in_plot} 🟡")
    num_cols = num_cols[~num_cols.duplicated()]
    
    fig, axes = plt.subplots(len(num_cols), 2,
                             figsize=(12, 3 * len(num_cols)),
                             squeeze=False)
    
    for i, col in enumerate(num_cols):
        # Share the y-axis across the row BEFORE plotting:
        # same scale on both sides, or the comparison lies.
        axes[i, 0].sharey(axes[i, 1])
    
        axes[i, 0].boxplot(train[col].dropna())
        axes[i, 0].set_title(f"{col} - TRAIN")
        axes[i, 0].grid(axis='y', alpha=0.7)
    
        axes[i, 1].boxplot(test[col].dropna())
        axes[i, 1].set_title(f"{col} - TEST")
        axes[i, 1].grid(axis='y', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

# ============================================================
# [7] OBJECT VALUES
# Four checks per text column: case/whitespace variants, rare values
# (typo strangers), numeric-in-disguise, test-only categories.
# Flags only - fixing happens in the treatment cell after inspection.
# Detection runs on RAW data; normalized views are used only where a
# comparison needs them, so evidence is never destroyed before it's seen.
# ============================================================
print("\n" + "[7] OBJECT VALUES")
print("-" * WIDTH)

RARE_PCT = 1       # categories rarer than this % of rows are typo suspects
NUMERIC_PCT = 90   # object column with more % convertible = numeric-in-disguise

train_cat_cols = train.select_dtypes(include='object').columns
test_cat_cols  = test.select_dtypes(include='object').columns

# Same guard philosophy as [6]: warn on mismatch, work on the common set
diff_cat = train_cat_cols.symmetric_difference(test_cat_cols)
if len(diff_cat) > 0:
    print("STATUS: WARNING - object column sets differ between train and test 🟡")
    print(f"  non-matching columns: {list(diff_cat)}\n")

cat_cols = train_cat_cols.intersection(test_cat_cols, sort=False)

if len(cat_cols) == 0:
    print("STATUS: PASS - no common object columns - section skipped 🟢")
else:
    # Check 3 first (numeric-in-disguise) so disguised numbers don't
    # pollute the categorical checks below with hundreds of fake "categories"
    numeric_obj_cols = []
    for col in cat_cols:
        valid_cnt   = train[col].notna().sum()
        numeric_cnt = pd.to_numeric(train[col], errors='coerce').notna().sum()
        if valid_cnt > 0 and (numeric_cnt / valid_cnt) * 100 >= NUMERIC_PCT:
            numeric_obj_cols.append(col)

    categorical_obj_cols = cat_cols.drop(numeric_obj_cols)

    if numeric_obj_cols:
        print(f"STATUS: WARNING - numeric-in-disguise (>= {NUMERIC_PCT}% convertible):")
        print(f"  {numeric_obj_cols}\n")

    any_issues = False
    for col in categorical_obj_cols:
        issues = []

        # Check 1: same category in different clothes.
        # train_norm is a VIEW for comparison only - train[col] stays raw.
        train_norm = train[col].dropna().astype(str).str.strip().str.lower()
        if train_norm.nunique() < train[col].nunique():
            variants = sorted(set(train[col].dropna().astype(str)) - set(train_norm.unique()))
            issues.append(f"case/whitespace variants: {variants[:3]}")

        # Check 2: typo strangers - real categories come in crowds,
        # typos come alone. Raw data, so variants stay visible as crumbs.
        vc = train[col].value_counts(normalize=True) * 100
        rare = vc[vc < RARE_PCT]
        if len(rare) > 0:
            issues.append(f"{len(rare)} rare values (<{RARE_PCT}%): "
                          f"{dict(rare.round(2).head(3))}")

        # Check 4: values test has but train never saw. Compared through
        # the view, so 'LONDON' vs 'London' doesn't false-alarm as unseen.
        test_norm = test[col].dropna().astype(str).str.strip().str.lower()
        unseen = set(test_norm.unique()) - set(train_norm.unique())
        if len(unseen) > 0:
            issues.append(f"{len(unseen)} test-only values: {list(unseen)[:3]}")

        if issues:
            any_issues = True
            print(f"[{col}]")
            for i in issues:
                print(f"   - {i}")

    if not any_issues and not numeric_obj_cols:
        print("STATUS: PASS - no object anomalies detected 🟢")

# ============================================================
# [8] DATASET REPORT
# PASS criteria:
#    1. Target exists (exactly once) in Train
#    2. No other Train-exclusive columns
#    3. No Test-exclusive columns
# ============================================================
print("\n" + "[8] DATASET REPORT")
print("-" * WIDTH)


# PASS only when the ONLY difference between the datasets is the target
# target_is_train_only = train_only.get(target) == 1
# if target_is_train_only and len(train_only) == 1 and not test_only:
#     print("STATUS: PASS  - columns match; target present only in Train 🟢")
# else:
# target_is_train_only = train_only.get(target) == 1
# no_other_train_columns = len(train_only) == 1
# no_test_only_columns = not test_only

# if target_is_train_only and no_other_train_columns and no_test_only_columns:
#     print("STATUS: PASS - Train/Test features match; target is present only in Train. 🟢")
# else:
#     print("STATUS: WARNING - Unexpected column differences detected. 🟡")
#     if not target_is_train_only:
#         print(f"  -> Target '{target}' issue in Train: {dict(train_only)}")
#     if not no_test_only_columns:
#         print(f"  -> Test-exclusive columns: {dict(test_only)}")
    
print("=" * WIDTH)

                                                          DATA INFORMATION REPORT                                                           

[1] DATA OVERVIEW
--------------------------------------------------------------------------------------------------------------------------------------------
Training dataset shape        : (800, 6)
Test dataset shape            : (300, 5)
Training features             : 6
Test features                 : 5

[2] COLUMN INFORMATION
--------------------------------------------------------------------------------------------------------------------------------------------
Training columns              : ['city', 'gender', 'age_text', 'segment', 'country', 'personality']
Test columns                  : ['city', 'gender', 'age_text', 'segment', 'country']

[3] COLUMN VALIDATION
--------------------------------------------------------------------------------------------------------------------------------------------
Features exclusive to Train   : {'p

In [ ]:
#SPLIT DATA
X_train = train.drop(columns=["Personality"])
y_train = train["Personality"]

X_test = test.reindex(columns=X_train.columns)

In [15]:
# 1 big data
train = pd.DataFrame({ 
    'age':             [25, 30, -5, 40, 150, 22, 35, 28],
    'height_cm':       [175, 180, 165, -20, 190, 172, 999, 168],
    'weight_kg':       [70, 85, 60, 75, -10, 68, 500, 72],
    'heart_rate':      [72, 80, 65, 90, -5, 75, 600, 68],
    'body_temp_c':     [36.6, 37.0, 35.8, 39.2, -100, 36.5, 150, 37.1],
    'sleep_hours':     [8, 7, 6, 9, -3, 7.5, 30, 8],
    'steps':           [5000, 8000, 3000, -100, 12000, 6500, 999999, 4000],
    'blood_pressure':  [120, 110, 130, -20, 250, 115, 0, 125],
    'years_experience':[3, 5, 1, -2, 40, 4, 150, 6],
    'salary':          [30000, 45000, 25000, -5000, 60000, 38000, 99999999, 42000]
})

test = pd.DataFrame({
    'age':              [19, 24, 32, 45, -8, 29, 200, 38],
    'height_cm':        [160, 172, 185, 178, 0, 169, 450, 181],
    'weight_kg':        [55, 68, 92, 80, -15, 73, 800, 65],
    'heart_rate':       [65, 78, 85, 72, -10, 95, 1000, 70],
    'body_temp_c':      [36.4, 37.1, 36.8, 38.5, -50, 36.2, 200, 37.0],
    'sleep_hours':      [7, 8, 6.5, 9, -4, 7.5, 48, 5],
    'steps':            [4000, 7500, 10000, 6000, -500, 12000, 2000000, 8500],
    'blood_pressure':   [110, 125, 118, 135, -30, 120, 500, 115],
    'years_experience': [0, 2, 8, 20, -3, 5, 250, 12],
    'salary':           [22000, 35000, 52000, 75000, -1000, 41000, 999999999, 60000]
})

In [71]:
# 2 small data
train = pd.DataFrame(
    [[1, 4], [2, 5], [3, 6], [10000, 7]],
    columns=['a', 'a']
)

test = pd.DataFrame(
    [[1, 4], [2, 5], [3, 10], [10000, 11]],
    columns=['a', 'a']
)